<a href="https://colab.research.google.com/github/Tushar-singhal12/-Intelligent-Task-Extraction-Categorization/blob/main/Text_Extraction_and_Categorization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import nltk
import numpy as np
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

text = "Rahul wakes up early every day. He goes to college in the morning and comes back at 3 pm. At present, Rahul is outside. He has to buy the snacks for all of us. Rahul should clean the room by 5 pm today."
# Create a Data Preprocessing function to remove special characters and to create tokens from data.
def preprocess_text(text):
    text = re.sub(r'[^a-zA-Z0-9 .]', '', text)  # Remove special characters
    return sent_tokenize(text)  # Sentence tokenization

# Extract responsible entity and deadline from the data
def extract_task_details(task_sentences):
    task_details = []

    for sent in task_sentences:
        words = word_tokenize(sent)
        pos_tags = pos_tag(words)
        entity = next((word for word, tag in pos_tags if tag in ["NNP", "PRP"]), None)

        # Improved deadline extraction
        deadline_patterns = re.findall(r'\b(?:by|before|on|at)?\s?(\d{1,2}\s?(?:AM|PM)|today|tomorrow|soon|next week|next month)\b', sent, re.IGNORECASE)
        deadline = " ".join(deadline_patterns) if deadline_patterns else None

        task_details.append({"task": sent, "assigned_to": entity, "deadline": deadline})

    return task_details
    return task_details
# Train Word2Vec model for embeddings
def train_word_embeddings(sentences):
    tokenized_sentences = [word_tokenize(sent.lower()) for sent in sentences]
    model = Word2Vec(tokenized_sentences, vector_size=50, window=5, min_count=1, workers=4)
    return model

# Categorize tasks using word embeddings and KMeans clustering
def categorize_tasks(task_sentences):
    model = train_word_embeddings(task_sentences)
    sentence_vectors = [np.mean([model.wv[word] for word in word_tokenize(sent.lower()) if word in model.wv], axis=0) for sent in task_sentences]

    # Handle cases where some sentences may not have any valid words
    sentence_vectors = [vec if vec is not None else np.zeros(50) for vec in sentence_vectors]

    num_clusters = min(len(task_sentences), 4)  # Adjusting clusters based on task count
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(sentence_vectors)

    predefined_categories = ["Household", "Work", "Errands", "Miscellaneous"]
    categorized_tasks = [
        {"task": task, "category": predefined_categories[label], "assigned_to": None, "deadline": None}
        for task, label in zip(task_sentences, cluster_labels)
    ]

    task_details = extract_task_details(task_sentences)
    for task in categorized_tasks:
        for detail in task_details:
            if task["task"] == detail["task"]:
                task["assigned_to"] = detail["assigned_to"]
                task["deadline"] = detail["deadline"]
                break

    return categorized_tasks

# Main Execution
text = '''
    Rahul has to buy snacks for all of us.
    She must complete the assignment by tomorrow.
    John should submit the report before 5 PM.
    I need to finish my homework.
    Rahul wakes up early every day. Rahul should clean the room by 5 pm today.'''
sentences = preprocess_text(text)
categorized_tasks = categorize_tasks(sentences)

# Output results
print("Structured List of Extracted Tasks:")
for task in categorized_tasks:
    print(task)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Structured List of Extracted Tasks:
{'task': '    Rahul has to buy snacks for all of us.', 'category': 'Household', 'assigned_to': 'Rahul', 'deadline': None}
{'task': 'She must complete the assignment by tomorrow.', 'category': 'Errands', 'assigned_to': 'She', 'deadline': 'tomorrow'}
{'task': 'John should submit the report before 5 PM.', 'category': 'Work', 'assigned_to': 'John', 'deadline': '5 PM'}
{'task': 'I need to finish my homework.', 'category': 'Household', 'assigned_to': 'I', 'deadline': None}
{'task': 'Rahul wakes up early every day.', 'category': 'Miscellaneous', 'assigned_to': 'Rahul', 'deadline': None}
{'task': 'Rahul should clean the room by 5 pm today.', 'category': 'Work', 'assigned_to': 'Rahul', 'deadline': '5 pm today'}
